In [ ]:
# SP-6 Phase 1: Feature Engineering Enhancement (PySpark Version)
# Adds strategic features for policy model

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
import time

spark = SparkSession.builder.getOrCreate()

print("=" * 80)
print("SP-6 Phase 1: Feature Engineering Enhancement (PySpark)")
print("=" * 80)
start_time = time.time()

In [ ]:
# Configuration
WORKSPACE_DIR = '/Workspace/Users/leo.lwakabamba@gmail.com/poker-ml-data/'
UC_VOLUME_DIR = '/Volumes/pokerml/default/data/'

# Input: SP-4 output (from UC Volume)
SP4_INPUT_PATH = UC_VOLUME_DIR + 'processed/sp4_features_complete'

# Output path (to UC Volume)
OUTPUT_PATH = UC_VOLUME_DIR + 'processed/sp6_features_enhanced'

print(f"Input: {SP4_INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
# Load SP-4 features
print("\n[1/4] Loading SP-4 features...")

# HARDCODED PATH
df = spark.read.parquet('/Volumes/pokerml/default/data/processed/sp4_features_complete')
initial_count = df.count()
print(f"   Loaded {initial_count:,} actions")
print(f"   Current features: {len(df.columns)} columns")

# ============================================================================
# TRACE PATTERN: Select a hand_id for pipeline validation
# ============================================================================
TRACE_HAND_ID = df.select('hand_id').first()['hand_id']
print(f"\n[TRACE] Selected TRACE_HAND_ID: {TRACE_HAND_ID}")

# TRACE: RAW INPUT from SP-4
print(f"\n[TRACE] RAW INPUT from SP-4 for {TRACE_HAND_ID}:")
trace_cols = ['hand_id', 'actor', 'street', 'action_type']
if 'position_from_button' in df.columns:
    trace_cols.append('position_from_button')
if 'position_name' in df.columns:
    trace_cols.append('position_name')
if 'hand_equity' in df.columns:
    trace_cols.append('hand_equity')
if 'predicted_strength' in df.columns:
    trace_cols.append('predicted_strength')
if 'starting_stack' in df.columns:
    trace_cols.append('starting_stack')
if 'target_profit_bb' in df.columns:
    trace_cols.append('target_profit_bb')
df.filter(F.col('hand_id') == TRACE_HAND_ID).select(trace_cols).show(20, truncate=False)

# Show available columns for reference
print(f"\n   Key columns available:")
key_cols = ['hand_equity', 'predicted_strength', 'opponent_strength_mean', 'target_profit_bb']
for col in key_cols:
    exists = col in df.columns
    print(f"      {col}: {'✓' if exists else 'MISSING'}")

In [ ]:
# Add strategic features for policy model
print("\n[2/4] Adding strategic features...")

# Ensure required columns exist with defaults
if 'starting_stack' not in df.columns:
    df = df.withColumn('starting_stack', F.lit(100.0))
if 'pot_before_action' not in df.columns:
    df = df.withColumn('pot_before_action', F.lit(1.0))
if 'facing_call' not in df.columns:
    df = df.withColumn('facing_call', F.lit(0.0))
if 'bb' not in df.columns:
    df = df.withColumn('bb', F.lit(1.0))
if 'hand_equity' not in df.columns:
    df = df.withColumn('hand_equity', F.lit(0.5))
if 'opponent_strength_mean' not in df.columns:
    df = df.withColumn('opponent_strength_mean', F.lit(0.5))
if 'opponents_active' not in df.columns:
    df = df.withColumn('opponents_active', F.lit(1))

# Stack-to-Pot Ratio (SPR)
print("   - Stack-to-pot ratio (SPR)")
df = df.withColumn(
    'spr',
    F.least(
        F.col('starting_stack') / (F.col('pot_before_action') + 0.000001),
        F.lit(100.0)
    )
)
df = df.withColumn('spr_low', F.when(F.col('spr') < 3, 1).otherwise(0))
df = df.withColumn('spr_medium', F.when((F.col('spr') >= 3) & (F.col('spr') <= 7), 1).otherwise(0))
df = df.withColumn('spr_high', F.when((F.col('spr') > 7) & (F.col('spr') <= 15), 1).otherwise(0))
df = df.withColumn('spr_very_deep', F.when(F.col('spr') > 15, 1).otherwise(0))

# Action Context
print("   - Action context features")
df = df.withColumn('can_check', F.when(F.col('facing_call') == 0, 1).otherwise(0))
df = df.withColumn(
    'pot_committed',
    F.least(
        F.col('facing_call') / (F.col('starting_stack') + 0.000001),
        F.lit(1.0)
    )
)
df = df.withColumn('pot_committed_heavy', F.when(F.col('pot_committed') > 0.33, 1).otherwise(0))

# Bet Sizing Context
print("   - Bet sizing features")
df = df.withColumn('raise_33_size', F.col('pot_before_action') * 0.33 + F.col('facing_call'))
df = df.withColumn('raise_75_size', F.col('pot_before_action') * 0.75 + F.col('facing_call'))
df = df.withColumn('raise_100_size', F.col('pot_before_action') * 1.0 + F.col('facing_call'))
df = df.withColumn('can_afford_raise_33', F.when(F.col('raise_33_size') <= F.col('starting_stack'), 1).otherwise(0))
df = df.withColumn('can_afford_raise_75', F.when(F.col('raise_75_size') <= F.col('starting_stack'), 1).otherwise(0))
df = df.withColumn('can_afford_raise_100', F.when(F.col('raise_100_size') <= F.col('starting_stack'), 1).otherwise(0))
df = df.withColumn(
    'raise_33_pct_stack',
    F.least(F.col('raise_33_size') / (F.col('starting_stack') + 0.000001), F.lit(1.0))
)
df = df.withColumn(
    'raise_75_pct_stack',
    F.least(F.col('raise_75_size') / (F.col('starting_stack') + 0.000001), F.lit(1.0))
)

# Remaining Stack Features
print("   - Remaining stack features")
df = df.withColumn('remaining_stack_after_call', F.col('starting_stack') - F.col('facing_call'))
df = df.withColumn('remaining_stack_bb', F.col('remaining_stack_after_call') / (F.col('bb') + 0.000001))
df = df.withColumn(
    'remaining_stack_pct',
    F.least(
        F.col('remaining_stack_after_call') / (F.col('starting_stack') + 0.000001),
        F.lit(1.0)
    )
)

# Position-Strength Interactions (using hand_equity - the TRUE hand strength)
print("   - Position-strength interactions")
df = df.withColumn('strength_advantage', F.col('hand_equity') - F.col('opponent_strength_mean'))
df = df.withColumn('strength_disadvantage', F.when(F.col('strength_advantage') < -0.1, 1).otherwise(0))
df = df.withColumn('strength_strong', F.when(F.col('strength_advantage') > 0.2, 1).otherwise(0))

# ============================================================================
# FIX: position_late check - use actual position_name values (BTN, CO, HJ)
# or position_bucket if it exists, checking for correct values
# Late positions in poker: BTN (Button), CO (Cutoff), HJ (Hijack)
# ============================================================================
if 'position_name' in df.columns:
    df = df.withColumn(
        'position_late',
        F.when(F.col('position_name').isin(['BTN', 'CO', 'HJ', 'button', 'late']), 1).otherwise(0)
    )
    print(f"   - position_late computed from position_name")
elif 'position_bucket' in df.columns:
    df = df.withColumn(
        'position_late',
        F.when(F.col('position_bucket').isin(['BTN', 'CO', 'HJ', 'late', 'button']), 1).otherwise(0)
    )
    print(f"   - position_late computed from position_bucket")
else:
    df = df.withColumn('position_late', F.lit(0))
    print(f"   - WARNING: No position column found, position_late = 0")

df = df.withColumn(
    'position_allows_bluff',
    F.when((F.col('position_late') == 1) & (F.col('opponents_active') <= 2), 1).otherwise(0)
)

# Game Context
print("   - Game context features")
df = df.withColumn('heads_up', F.when(F.col('opponents_active') == 1, 1).otherwise(0))
df = df.withColumn('three_way', F.when(F.col('opponents_active') == 2, 1).otherwise(0))
df = df.withColumn('multiway', F.when(F.col('opponents_active') >= 3, 1).otherwise(0))

# Opponent Threats
print("   - Opponent threat features")
if 'opponent_nutted_count' in df.columns:
    df = df.withColumn('facing_nutted_opponent', F.when(F.col('opponent_nutted_count') > 0, 1).otherwise(0))
    if 'opponent_middle_count' in df.columns:
        df = df.withColumn(
            'facing_multiple_strong',
            F.when(F.col('opponent_nutted_count') + F.col('opponent_middle_count') >= 2, 1).otherwise(0)
        )
    else:
        df = df.withColumn('facing_multiple_strong', F.lit(0))
else:
    df = df.withColumn('facing_nutted_opponent', F.lit(0))
    df = df.withColumn('facing_multiple_strong', F.lit(0))

# Pot Odds
print("   - Enhanced pot odds")
if 'pot_odds_call' in df.columns:
    df = df.withColumn('pot_odds_favorable', F.when(F.col('pot_odds_call') < 0.25, 1).otherwise(0))
    df = df.withColumn('pot_odds_unfavorable', F.when(F.col('pot_odds_call') > 0.40, 1).otherwise(0))
else:
    df = df.withColumn('pot_odds_favorable', F.lit(0))
    df = df.withColumn('pot_odds_unfavorable', F.lit(0))

# Street-specific SPR
print("   - Street-specific SPR flags")
df = df.withColumn(
    'postflop_low_spr',
    F.when((F.col('street').isin(['flop', 'turn'])) & (F.col('spr') < 5), 1).otherwise(0)
)
df = df.withColumn(
    'preflop_deep_stack',
    F.when((F.col('street') == 'preflop') & (F.col('spr') > 20), 1).otherwise(0)
)

print("   Strategic features added")

# TRACE: After adding strategic features
print(f"\n[TRACE] After strategic features for {TRACE_HAND_ID}:")
df.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'actor', 'street',
    'spr', 'spr_low', 'spr_high',
    'can_check', 'pot_committed',
    'strength_advantage', 'heads_up', 'multiway',
    'position_late'
).show(10, truncate=False)

In [ ]:
# Encode categorical features manually (NO SparkML - avoids 100MB model limit)
print("\n[3/4] Encoding categorical features...")

categorical_features = ['position_bucket', 'predicted_bucket']

# Only encode columns that exist
existing_categorical = [col for col in categorical_features if col in df.columns]

if existing_categorical:
    print(f"   Encoding: {', '.join(existing_categorical)}")
    
    for col in existing_categorical:
        # Fill nulls
        df = df.withColumn(col, F.coalesce(F.col(col), F.lit('unknown')))
        
        # Get distinct values
        distinct_values = [row[0] for row in df.select(col).distinct().collect()]
        print(f"      {col} values: {distinct_values}")
        
        # Create dummy columns manually (NO StringIndexer - avoids ML model size limit)
        for val in distinct_values[:10]:  # Limit to top 10 values
            safe_val = str(val).replace(' ', '_').replace('-', '_')
            df = df.withColumn(
                f"{col}_{safe_val}",
                F.when(F.col(col) == val, 1).otherwise(0)
            )
        
        # Drop original string column to avoid issues downstream
        df = df.drop(col)
else:
    print("   No categorical features to encode")

print(f"   Features after encoding: {len(df.columns)}")

In [ ]:
# Save enhanced dataset
print("\n[4/4] Saving enhanced dataset...")

# Drop temporary columns
drop_cols = ['raise_33_size', 'raise_75_size', 'raise_100_size']
df = df.drop(*[c for c in drop_cols if c in df.columns])

# TRACE: Final output before save
print(f"\n[TRACE] FINAL OUTPUT for {TRACE_HAND_ID}:")
df.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'actor', 'street', 'action_type',
    'spr', 'can_check', 'pot_committed',
    'strength_advantage', 'position_late'
).show(10, truncate=False)

# Save as Parquet - HARDCODED PATH
df.write.mode('overwrite').parquet('/Volumes/pokerml/default/data/processed/sp6_features_enhanced')

# Get final stats
final_count = df.count()
final_cols = len(df.columns)

elapsed = time.time() - start_time

print("\n" + "=" * 80)
print("SP-6 Phase 1 COMPLETE (PySpark)!")
print("=" * 80)
print(f"\nRuntime: {elapsed:.1f} seconds")
print(f"Dataset: {final_count:,} actions, {final_cols} features")
print(f"\nData Lineage:")
print(f"  Input: SP-4 output (/Volumes/pokerml/default/data/processed/sp4_features_complete)")
print(f"  Output: /Volumes/pokerml/default/data/processed/sp6_features_enhanced")
print(f"\n[TRACE] TRACE_HAND_ID used: {TRACE_HAND_ID}")
print(f"\nNext step: Run SP-7 (07_LabelCreation) to generate action labels")

In [ ]:
# Show sample of new features
print("\nSample of new strategic features:")
df.select(
    'hand_id', 'street',
    'spr', 'spr_low', 'spr_high',
    'can_check', 'pot_committed',
    'strength_advantage', 'heads_up', 'multiway'
).show(10, truncate=False)

---
## OLD PANDAS VERSION (Commented Out)
The code below is the original pandas-based implementation.
Kept for reference.

In [ ]:
# # OLD VERSION - PANDAS IMPLEMENTATION
# # ====================================
# 
# # SP-6 Phase 1: Feature Engineering Enhancement
# # Adds strategic features for policy model
# 
# import pandas as pd
# import numpy as np
# import time
# from pathlib import Path
# 
# # ... (rest of original pandas implementation)
# # See git history for full original code